In [5]:
import tkinter as tk
from tkinter import filedialog, ttk, messagebox
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg


class ForensicDNA:
    
    def __init__(self, root):
        self.root = root
        self.root.title("FORENSIC DNA ANALYSIS SYSTEM")
        self.root.geometry("1100x700")
        self.root.config(bg="#e8f0f7")

        self.df = None

        tk.Label(root, text="FORENSIC DNA ANALYSIS",
                 font=("Arial", 26, "bold"),
                 bg="#e8f0f7", fg="#222").pack(pady=20)

        # Frame for buttons
        frame = tk.Frame(root, bg="#e8f0f7")
        frame.pack()

        tk.Button(frame, text="Load Dataset", width=18, font=("Arial", 12),
                  bg="#4caf50", fg="white", command=self.load_dataset).grid(row=0, column=0, padx=10, pady=10)

        tk.Button(frame, text="Compare DNA", width=18, font=("Arial", 12),
                  bg="#673ab7", fg="white", command=self.compare_samples).grid(row=0, column=1, padx=10, pady=10)

        # Dropdowns for suspect and evidence
        self.suspect_var = tk.StringVar()
        self.evidence_var = tk.StringVar()

        tk.Label(root, text="Choose Suspect:", font=("Arial", 14), bg="#e8f0f7").pack()
        self.suspect_combo = ttk.Combobox(root, textvariable=self.suspect_var,
                                          width=50, state="readonly", font=("Arial", 12))
        self.suspect_combo.pack(pady=5)

        tk.Label(root, text="Choose Evidence Sample:", font=("Arial", 14), bg="#e8f0f7").pack()
        self.evidence_combo = ttk.Combobox(root, textvariable=self.evidence_var,
                                           width=50, state="readonly", font=("Arial", 12))
        self.evidence_combo.pack(pady=5)

        # Output box
        self.output = tk.Text(root, width=120, height=18, font=("Arial", 11))
        self.output.pack(pady=15)

    # ---------------------- LOAD DATASET ----------------------
    def load_dataset(self):
        file = filedialog.askopenfilename(title="Select DNA Dataset",
                                          filetypes=[("CSV Files", "*.csv")])
        if file:
            self.df = pd.read_csv(file)
            names = list(self.df["Sample Name"])

            self.suspect_combo["values"] = names
            self.evidence_combo["values"] = names

            messagebox.showinfo("Success", "Dataset Loaded Successfully!")

    # ----------------------- FIND LOCI PAIRS ----------------------
    def get_loci_pairs(self):
        loci = {}
        for col in self.df.columns:
            if col not in ["Sample Name"] and "." not in col:
                allele1 = col
                allele2 = col + ".1"

                if allele2 in self.df.columns:
                    loci[col] = (allele1, allele2)

        return loci

    # ----------------------- COMPARE SAMPLES ----------------------
    def compare_samples(self):
        if self.df is None:
            messagebox.showerror("Error", "Load dataset first!")
            return

        suspect = self.suspect_var.get()
        evidence = self.evidence_var.get()

        if suspect == "" or evidence == "":
            messagebox.showerror("Error", "Select both Suspect and Evidence!")
            return

        suspect_row = self.df[self.df["Sample Name"] == suspect].iloc[0]
        evidence_row = self.df[self.df["Sample Name"] == evidence].iloc[0]

        loci = self.get_loci_pairs()

        total_score = 0
        max_score = len(loci) * 2

        labels = []
        scores = []

        result = f"DNA ANALYSIS REPORT\nSuspect: {suspect}\nEvidence: {evidence}\n\n"
        result += "="*70 + "\n"

        for locus, (a1, a2) in loci.items():

            s1, s2 = suspect_row[a1], suspect_row[a2]
            e1, e2 = evidence_row[a1], evidence_row[a2]

            score = 0
            if s1 == e1: score += 1
            if s2 == e2: score += 1

            total_score += score
            scores.append(score)
            labels.append(locus)

            result += f"{locus:<10} Suspect: ({s1}, {s2})   Evidence: ({e1}, {e2})   Score: {score}\n"

        similarity = (total_score / max_score) * 100
        result += "\n" + "="*70 + "\n"
        result += f"Similarity Score: {similarity:.2f}%\n"

        self.output.delete(1.0, tk.END)
        self.output.insert(tk.END, result)

        self.plot_graph(labels, scores)

    # ------------------------ GRAPH ------------------------
    def plot_graph(self, labels, scores):
        win = tk.Toplevel()
        win.title("STR Match Score Graph")

        fig = plt.Figure(figsize=(8, 4))
        ax = fig.add_subplot(111)

        ax.bar(labels, scores)
        ax.set_ylabel("Match Score (0 - 2)")
        ax.set_title("Suspect vs Evidence STR Comparison")
        ax.tick_params(axis="x", rotation=90)

        canvas = FigureCanvasTkAgg(fig, master=win)
        canvas.get_tk_widget().pack()
        canvas.draw()


root = tk.Tk()
app = ForensicDNA(root)
root.mainloop()
